# 12.3 Multiprocessing

**Prerequisites:** 12.1 Concurrency, Parallelism and the GIL, 12.2 Threading  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Processes vs threads: separate memory, separate GIL, real parallelism
- 🔴 **`spawn` vs `fork`** - and why Windows and macOS behave differently
- 🔴 **`if __name__ == "__main__":`** - not style, a hard requirement
- `Process`, `Pool`, and `ProcessPoolExecutor`
- Measuring the speedup on CPU-bound work - and the startup cost that eats it
- Talking between processes: `Queue`, `Pipe`, `Value`, `Array`
- `multiprocessing.shared_memory` for large data
- 🔴 What can and cannot be pickled
- When multiprocessing is *not* worth it

---

## Why processes

From **12.1**: threads cannot run Python bytecode in parallel, because of the GIL. Processes can — **each has its own interpreter and its own GIL**.

```
   THREADS (12.2)                        PROCESSES (here)
   ┌──────────────────────────┐          ┌────────┐  ┌────────┐  ┌────────┐
   │ shared memory, one GIL   │          │ own    │  │ own    │  │ own    │
   │  ┌────┐ ┌────┐ ┌────┐    │          │ memory │  │ memory │  │ memory │
   │  │ T1 │ │ T2 │ │ T3 │    │          │ own GIL│  │ own GIL│  │ own GIL│
   │  └────┘ └────┘ └────┘    │          └────────┘  └────────┘  └────────┘
   └──────────────────────────┘             pickled messages between them
        take turns                             genuinely parallel
```

What you gain: **real use of every core**. What you pay:

| Cost | Detail |
|---|---|
| Startup | ~50 ms per process on Windows, vs ~50 µs for a thread |
| Memory | a full interpreter each |
| Communication | everything is **pickled**, copied, and unpickled |
| Debugging | tracebacks come from another process |

That third row is the one that decides most designs: sending a large object to a worker can cost more than the work itself.

## 🔴 `spawn` vs `fork` - the difference that breaks notebooks

How a child process comes into existence differs by platform, and it is not a detail.

| | `fork` | `spawn` | `forkserver` |
|---|---|---|---|
| Default on | Linux (3.13 and earlier) | **Windows, macOS**, Linux from **3.14** | Linux, opt-in |
| How | clones the whole process | starts a fresh interpreter | forks from a clean helper |
| Child starts with | a copy of everything | **nothing** - it re-imports your module | a minimal copy |
| Speed | fast | slow | medium |
| Safe with threads/locks | 🔴 **no** | yes | yes |

> ### Version note - the default changed in 3.14
> Linux used `fork` for years, which is fast but genuinely unsafe: forking a process that holds a lock gives the child a copy of that lock, permanently locked, with no thread to release it. **Python 3.14 changed the Linux default to `forkserver`** for exactly this reason. Code that silently depended on inheriting state through `fork` will break — so set the method explicitly if you care.

### What `spawn` means for you

The child **re-imports the module that started it**. So the worker function must be importable, and anything at module level runs **again, in every child**.

## 🔴 The `__main__` guard is mandatory

```
    if __name__ == "__main__":
        with ProcessPoolExecutor() as pool:
            ...
```

Without it, under `spawn`:

1. Parent starts, reaches the pool line, spawns a child
2. Child re-imports the module — and reaches the pool line too
3. Child spawns its own children
4. Repeat until the machine gives up

The guard works because the child imports your file under the name **`__mp_main__`**, not `__main__` — so the guarded block is skipped in children while function and class definitions above it still get created.

The first cell below proves that, by printing `__name__` and the process id in both.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
import time
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py123_"))
print("scratch:", WORK)


def run_script(name: str, source: str, timeout: int = 180) -> str:
    """Write a script and run it in a fresh interpreter, returning stdout.

    🔴 Why not just create a Pool in this cell? Because a notebook cell is
    not an importable module. Under `spawn` the child re-imports the
    parent's __main__ to rebuild the target function - and a function that
    exists only in a cell's namespace cannot be found, so the pool dies
    with BrokenProcessPool. Running a real script is the honest fix, and
    it puts the __main__ guard where you would actually write it.
    """
    path = WORK / name
    path.write_text(textwrap.dedent(source), encoding="utf-8")
    finished = subprocess.run(
        [sys.executable, str(path)],
        capture_output=True, text=True, timeout=timeout,
    )
    if finished.returncode != 0:
        return f"[exit {finished.returncode}]\n{finished.stderr.strip()[:800]}"
    return finished.stdout.rstrip()


print("helper ready")

In [ ]:
print(run_script("guard_demo.py", '''
    import multiprocessing as mp
    import os

    # This line is at module level, so it runs in EVERY process.
    print(f"  module body: pid={os.getpid():<6} __name__={__name__!r}")

    def worker():
        print(f"  worker     : pid={os.getpid():<6} __name__={__name__!r}")

    if __name__ == "__main__":
        print("  guarded block runs ONLY in the parent")
        process = mp.get_context("spawn").Process(target=worker)
        process.start()
        process.join()
'''))

print()
print("Two different pids, and the child's __name__ is '__mp_main__'.")
print("That is precisely why the guard works: the child re-imports the")
print("file (so `worker` gets defined) but skips the guarded block.")

## Does it actually go faster?

**12.1** measured threads on CPU-bound work and got **no** speedup. Same workload, same machine, now with processes.

The script below times three versions of the same prime counting: serial, threads, and processes.

In [ ]:
print(run_script("speedup.py", '''
    import os
    import time
    from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor


    def count_primes(limit):
        found = 0
        for candidate in range(2, limit):
            divisor, is_prime = 2, True
            while divisor * divisor <= candidate:
                if candidate % divisor == 0:
                    is_prime = False
                    break
                divisor += 1
            if is_prime:
                found += 1
        return found


    if __name__ == "__main__":
        LIMIT, TASKS = 250_000, 4
        print(f"  cpu cores: {os.cpu_count()}   tasks: {TASKS}")

        started = time.perf_counter()
        [count_primes(LIMIT) for _ in range(TASKS)]
        serial = time.perf_counter() - started
        print(f"  serial    {serial:6.2f}s")

        started = time.perf_counter()
        with ThreadPoolExecutor(max_workers=TASKS) as pool:
            list(pool.map(count_primes, [LIMIT] * TASKS))
        threaded = time.perf_counter() - started
        print(f"  threads   {threaded:6.2f}s   {serial / threaded:5.2f}x")

        started = time.perf_counter()
        with ProcessPoolExecutor(max_workers=TASKS) as pool:
            list(pool.map(count_primes, [LIMIT] * TASKS))
        processes = time.perf_counter() - started
        print(f"  processes {processes:6.2f}s   {serial / processes:5.2f}x")
'''))

print()
print("Threads: no better than serial - the GIL, exactly as in 12.1.")
print("Processes: a real speedup, because each has its own interpreter.")

### 🔴 The startup cost is not small

A process pool is not free. Under `spawn` each worker is a **fresh interpreter** that must start and re-import your module — tens of milliseconds each, and more if your module imports something heavy.

So there is a threshold: below it, the pool costs more than it saves. The script below finds roughly where that threshold sits, by running the same tiny task in both.

In [ ]:
print(run_script("overhead.py", '''
    import time
    from concurrent.futures import ProcessPoolExecutor


    def tiny(n):
        return n * n


    def chunky(n):
        total = 0
        for i in range(2_000_000):
            total += i % 7
        return total


    if __name__ == "__main__":
        started = time.perf_counter()
        with ProcessPoolExecutor(max_workers=4) as pool:
            pool.submit(tiny, 1).result()
        print(f"  starting a 4-worker pool and doing nothing: "
              f"{time.perf_counter() - started:5.2f}s")

        for label, func, tasks in (("trivial", tiny, 8), ("heavy", chunky, 8)):
            started = time.perf_counter()
            [func(i) for i in range(tasks)]
            serial = time.perf_counter() - started

            started = time.perf_counter()
            with ProcessPoolExecutor(max_workers=4) as pool:
                list(pool.map(func, range(tasks)))
            parallel = time.perf_counter() - started

            verdict = "WORTH IT" if parallel < serial else "NOT worth it"
            print(f"  {label:<8} serial {serial:6.2f}s  pool {parallel:6.2f}s  "
                  f"-> {verdict}")
'''))

print()
print("🔴 For small tasks the pool is dramatically SLOWER. Parallelism has a")
print("   fixed cost, and it has to be earned. Always measure before assuming.")

## Talking between processes

Separate memory means no shared variables. Everything must be **pickled**, sent, and unpickled.

| Tool | Shape | Use for |
|---|---|---|
| `Queue` | many-to-many | work distribution, results |
| `Pipe` | two ends, point-to-point | a fast private channel |
| `Value` / `Array` | shared C types | a counter, a small fixed array |
| `Manager` | shared `dict`/`list` proxies | convenience; slow |
| `shared_memory` | a raw block of bytes | large arrays, zero-copy |

🔴 `Value` and `Array` are shared, so they need locking just like **12.2** — and they come with a lock built in for exactly that reason.

In [ ]:
print(run_script("ipc.py", '''
    import multiprocessing as mp


    def producer(queue, count):
        for i in range(count):
            queue.put(f"job-{i}")
        queue.put(None)                      # sentinel, as in 12.2


    def consumer(queue, results):
        while True:
            item = queue.get()
            if item is None:
                break
            results.put(item.upper())
        results.put(None)


    def send_status(connection):
        # 🔴 Deliberately NOT a lambda. A Process target is pickled by
        # qualified name, so it must be findable when the child re-imports
        # this module. An earlier draft of this cell used a lambda here and
        # died with: AttributeError: module '__main__' has no attribute '<lambda>'
        connection.send({"status": "ok", "checked": 3})
        connection.close()


    def increment(counter, times):
        for _ in range(times):
            with counter.get_lock():         # 🔴 shared state still needs a lock
                counter.value += 1


    if __name__ == "__main__":
        ctx = mp.get_context("spawn")

        # ---- Queue: many-to-many ----
        jobs, done = ctx.Queue(), ctx.Queue()
        p1 = ctx.Process(target=producer, args=(jobs, 5))
        p2 = ctx.Process(target=consumer, args=(jobs, done))
        p1.start(); p2.start()
        collected = []
        while True:
            item = done.get(timeout=30)
            if item is None:
                break
            collected.append(item)
        p1.join(timeout=30); p2.join(timeout=30)
        print("  Queue    ->", collected)

        # ---- Pipe: point-to-point ----
        left, right = ctx.Pipe()
        sender = ctx.Process(target=send_status, args=(right,))
        sender.start()
        print("  Pipe     ->", left.recv())
        sender.join(timeout=30)
        left.close()

        # ---- Value, with its built-in lock ----
        counter = ctx.Value("i", 0)
        workers = [ctx.Process(target=increment, args=(counter, 10_000))
                   for _ in range(4)]
        for w in workers:
            w.start()
        for w in workers:
            w.join(timeout=60)
        print(f"  Value    -> {counter.value:,} (expected 40,000)")

        # ---- and what a process CANNOT see ----
        shared_list = []

        def appender(target):
            target.append("from child")

        child = ctx.Process(target=appender, args=(shared_list,))
        child.start(); child.join(timeout=30)
        print(f"  plain list after the child appended: {shared_list}")
        print("           ^ empty. The child mutated its own COPY.")
'''))

### `shared_memory` - when copying is the bottleneck

Sending a 100 MB array to four workers via a `Queue` pickles and copies it four times. `multiprocessing.shared_memory` (3.8+) instead gives every process a window onto the **same physical bytes** — no copy at all.

🔴 It is unmanaged memory. Every process must `close()` its view, and exactly one must `unlink()` to release the block — otherwise it leaks until reboot on some platforms.

In [ ]:
print(run_script("sharedmem.py", '''
    import time
    from multiprocessing import Process
    from multiprocessing import shared_memory


    def square_in_place(name, start, stop):
        existing = shared_memory.SharedMemory(name=name)
        view = memoryview(existing.buf).cast("i")
        for i in range(start, stop):
            view[i] = view[i] * 2
        view.release()
        existing.close()                     # every process closes its view


    if __name__ == "__main__":
        COUNT = 400_000
        block = shared_memory.SharedMemory(create=True, size=COUNT * 4)
        data = memoryview(block.buf).cast("i")
        for i in range(COUNT):
            data[i] = i

        print(f"  block: {block.size:,} bytes, name={block.name}")
        print(f"  before: {[data[i] for i in range(5)]}")

        started = time.perf_counter()
        half = COUNT // 2
        workers = [
            Process(target=square_in_place, args=(block.name, 0, half)),
            Process(target=square_in_place, args=(block.name, half, COUNT)),
        ]
        for w in workers:
            w.start()
        for w in workers:
            w.join(timeout=60)

        print(f"  after : {[data[i] for i in range(5)]}")
        print(f"  last  : {data[COUNT - 1]:,} (was {COUNT - 1:,})")
        print(f"  time  : {time.perf_counter() - started:.2f}s, nothing copied")

        data.release()
        block.close()
        block.unlink()                       # 🔴 exactly one process unlinks
        print("  released")
'''))

## 🔴 What cannot be pickled

Everything crossing a process boundary is pickled. Plenty of ordinary Python is not picklable, and you find out at `submit()` time.

| Picklable | Not picklable |
|---|---|
| module-level functions and classes | **lambdas** |
| functions defined inside `if __name__ == "__main__":` | functions defined **inside another function** |
| built-in types, most containers | closures |
| `dataclass` instances of picklable fields | open files, sockets, database connections |
| `numpy` arrays | generators, thread locks, `threading.Lock` |

Pickle stores a function by **qualified name**, not by its code — which is why anything it cannot look up again fails, and why the child must be able to import the module. The workaround for a lambda is a module-level function; for an unpicklable resource, create it **inside** the worker rather than passing it in — which is the right design anyway, since a database connection cannot be shared across processes.

In [ ]:
print(run_script("pickling.py", '''
    import pickle
    import threading


    def module_level(n):
        return n * 2


    def make_multiplier(factor):
        """Returns a closure - defined INSIDE another function."""
        def truly_nested(n):
            return n * factor
        return truly_nested


    if __name__ == "__main__":
        def inside_the_guard(n):
            # Still MODULE scope - the guard is not a function.
            return n * 2

        handle = open(__file__)
        candidates = [
            ("module-level function", module_level),
            ("defined inside the guard", inside_the_guard),
            ("defined inside a function", make_multiplier(2)),
            ("lambda", lambda n: n * 2),
            ("dict", {"a": 1}),
            ("open file", handle),
            ("threading.Lock", threading.Lock()),
            ("generator", (x for x in range(3))),
        ]

        for label, obj in candidates:
            try:
                pickle.dumps(obj)
                print(f"  {label:<26} OK")
            except Exception as exc:
                print(f"  {label:<26} {type(exc).__name__}")

        handle.close()
'''))

print()
print("Note the distinction: 'inside the guard' pickles fine, because the")
print("guard is not a function - the name still lives at module level and")
print("pickle can look it up. Only a function defined inside ANOTHER")
print("function has no importable name.")
print()
print("This is why worker functions must live at module level - and why a")
print("notebook cell cannot supply one to a spawn-based pool.")

## When multiprocessing is the wrong answer

| Situation | Better |
|---|---|
| I/O-bound work | threads (**12.2**) or `asyncio` (**12.5**) |
| Tasks shorter than ~50 ms | do it serially; the startup dominates |
| Large data per task | shared memory, or move the work to the data |
| Numeric arrays | NumPy/SciPy already release the GIL and use optimised kernels |
| One machine is not enough | a task queue - Celery, RQ, Dask |

### The pragmatic order

1. **Make it faster serially first.** A better algorithm (**14**) beats parallelising a bad one, and it beats it on every machine.
2. **Check whether it is really CPU-bound** (**12.1**).
3. **Use `ProcessPoolExecutor`** (**12.4**) rather than raw `Process` objects.
4. **Measure.** Every time.

> Amdahl's law, informally: if 20% of your program is inherently serial, then even with infinite cores you cannot beat a 5x speedup. Parallelism raises the ceiling; it does not remove it.

In [ ]:
# ---- tidy up ----
import multiprocessing

shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed :", not WORK.exists())
print("child processes :", multiprocessing.active_children() or "none")
print("\nEvery example ran in its own interpreter via subprocess, so nothing")
print("was left behind in this one.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Omitting `if __name__ == "__main__":`.** Under `spawn`, every child re-imports your module and spawns its own children. It is not style - it is required.
2. 🔴 **Passing a lambda or a nested function to a pool.** It cannot be pickled. Use a module-level function.
3. 🔴 **Expecting a notebook cell's function to work in a process pool.** The child re-imports `__main__`, and a cell is not importable. Put workers in a `.py` file.
4. **Assuming a pool is always faster.** For small tasks the startup cost dominates and it is far slower.
5. **Sharing a database connection or open file with a worker.** Not picklable, and not safe. Create it inside the worker.
6. **Mutating a global and expecting the parent to see it.** Separate memory - the child changed its own copy.
7. **Forgetting that `Value`/`Array` still race.** Use `get_lock()`.
8. **Leaking `shared_memory`.** Every process `close()`s; exactly one `unlink()`s.
9. **Relying on `fork` semantics.** macOS and Windows use `spawn`, and 3.14 changed the Linux default to `forkserver`.

## Best Practices

- Put worker functions at module level, in an importable module.
- Always guard the entry point with `if __name__ == "__main__":`.
- Prefer `ProcessPoolExecutor` (**12.4**) to managing `Process` objects.
- Choose the start method explicitly with `mp.get_context(...)` rather than inheriting a platform default that changed in 3.14.
- Chunk work so each task is substantial - hundreds of milliseconds, not microseconds.
- Keep the data crossing the boundary small; use `shared_memory` when it cannot be.
- Create unpicklable resources inside the worker, not in the parent.
- Measure serial first, then parallel, and keep the serial version if it wins.

## Practice Exercises

Try these before moving on.

1. Run the speedup script with `TASKS` set to twice your core count. Does the speedup keep improving? Why not?
2. Find the crossover point on your machine: shrink `chunky` until the pool stops being worth it. Roughly how much work does a task need to justify a process?
3. 🔴 Write a script that creates a `ProcessPoolExecutor` **without** the `__main__` guard, and run it with a hard timeout and a process monitor open. Predict what happens first.
4. Convert the `Queue` example to `ProcessPoolExecutor.map` and compare the amount of code.
5. Pass a 50 MB `bytes` object to four workers via a pool, then via `shared_memory`. Time both and explain the gap.
6. Take a `dataclass` (**5.3**) with a `threading.Lock` field and try to pickle it. Fix it with `__getstate__`/`__setstate__`.
7. Compare `mp.get_context('spawn')` with `'fork'` (on Linux/macOS) for a worker that reads a module-level global set before the fork. Which sees the value, and why is relying on that a trap?